In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import drive

# =====================================================================
# 1. CONFIGURAZIONE PERCORSI E MONTAGGIO GOOGLE DRIVE
# =====================================================================
drive.mount('/content/drive')

extraction_path = '/content/drive/MyDrive/Colab Notebooks/sistemi operativi/dataset_braille'
voc_root = os.path.join(extraction_path, 'character_label', 'voc-data')
annotations_dir = os.path.join(voc_root, 'Annotations')
images_dir = os.path.join(voc_root, 'JPEGImages')

# Parametri ottimizzati per STM32
IMG_H, IMG_W = 32, 128
GRID_CELLS = 5
NUM_CLASSES = 65

In [ ]:
import tensorflow as tf

def wide_separable_attention_block(input_tensor, channels):
    # Attenzione riproporzionata sui canali originali
    attn = tf.keras.layers.Conv2D(channels // 4, kernel_size=3, padding='same')(input_tensor)
    attn = tf.keras.layers.BatchNormalization()(attn)
    attn = tf.keras.layers.Activation('relu')(attn)

    attn = tf.keras.layers.Conv2D(1, kernel_size=1, padding='same', activation='sigmoid')(attn)
    return tf.keras.layers.Multiply()([input_tensor, attn])

def create_wide_separable_braille_net(num_classes=64):
    inputs = tf.keras.Input(shape=(32, 32, 1), name='image_input')

    # Ripristiniamo la progressione originale dei canali: 16 -> 32 -> 64 -> 128
    # Ma usiamo SeparableConv2D per tagliare il peso dei parametri
    x = tf.keras.layers.SeparableConv2D(16, (3, 3), strides=(2, 2), padding='same', use_bias=False)(inputs) # 16x16
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    x = tf.keras.layers.SeparableConv2D(32, (3, 3), strides=(2, 2), padding='same', use_bias=False)(x)      # 8x8
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    # Pixel Attention tarata sui 32 canali (esattamente come nel tuo script di partenza)
    x = wide_separable_attention_block(x, 32)

    x = tf.keras.layers.SeparableConv2D(64, (3, 3), strides=(2, 2), padding='same', use_bias=False)(x)      # 4x4
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    x = tf.keras.layers.SeparableConv2D(128, (3, 3), strides=(1, 1), padding='same', use_bias=False)(x)     # Rimane 4x4 per elaborare i 128 canali
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    # Global Average Pooling restituisce il vettore originale da 128 elementi (stabilità matematica per le 64 classi)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # Teste di output
    class_output = tf.keras.layers.Dense(num_classes, activation=None, name='final_class_output')(x)
    box_conf = tf.keras.layers.Dense(1, activation='sigmoid', name='final_conf_output')(x)

    model = tf.keras.Model(inputs=inputs, outputs=[class_output, box_conf], name="STM32_Braille_WideSeparableNet")
    return model

# Inizializzazione della rete a canali larghi ma separabili
model = create_wide_separable_braille_net(num_classes=65)

model.summary()

In [ ]:
import xml.etree.ElementTree as ET
import numpy as np
import tensorflow as tf

# =====================================================================
# 3. BLOCCO 3 CORAZZATO: ISPEZIONE DIRETTA E RITAGLIO MIRATO 48x48
# =====================================================================
def load_narrow_patch_dataset_inspected(images_dir, annotations_dir, patch_size=32):
    window_images = []
    window_classes = []
    window_confs = []
    window_filenames = []

    # 1. Controllo di esistenza fisico delle cartelle
    if not os.path.exists(annotations_dir):
        print(f"[ERRORE FISICO] La cartella annotazioni NON ESISTE al percorso: {annotations_dir}")
        return np.array([]), np.array([]), np.array([]), []
    if not os.path.exists(images_dir):
        print(f"[ERRORE FISICO] La cartella immagini NON ESISTE al percorso: {images_dir}")
        return np.array([]), np.array([]), np.array([]), []

    all_files = os.listdir(annotations_dir)
    ann_files = [f for f in all_files if f.lower().endswith('.xml')]
    ann_files = sorted(ann_files)

    print(f"[INFO] Trovati {len(ann_files)} file .xml nella cartella.")
    if len(ann_files) == 0:
        print("[ERRORE] La cartella XML è vuota o l'estensione non è .xml!")
        return np.array([]), np.array([]), np.array([]), []

    # Prendiamo il raggio del quadratino (es. 48 -> 24 pixel dal centro)
    half_size = patch_size // 2

    for ann_file in ann_files:
        ann_path = os.path.join(annotations_dir, ann_file)

        # Rimuoviamo il try/except generico per vedere l'errore reale se il codice si rompe
        tree = ET.parse(ann_path)
        root = tree.getroot()

        # Risoluzione nome file immagine
        img_filename_elem = root.find('filename')
        img_filename = img_filename_elem.text.strip() if img_filename_elem is not None else ann_file.lower().replace('.xml', '.jpg')

        img_path = os.path.join(images_dir, img_filename)

        # Tentativi di recupero se l'estensione differisce (es. .jpg vs .JPG)
        if not os.path.exists(img_path):
            base_name = os.path.splitext(img_filename)[0]
            for ext in ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
                test_path = os.path.join(images_dir, base_name + ext)
                if os.path.exists(test_path):
                    img_path = test_path
                    break

        if not os.path.exists(img_path):
            # Stampiamo un avviso se manca l'immagine corrispondente, senza bloccare il programma
            print(f"[AVVISO] Manca l'immagine per l'XML: {ann_file} (Cercata in: {img_path})")
            continue

        # Caricamento nativo dell'immagine in scala di grigi
        img = tf.keras.preprocessing.image.load_img(img_path, color_mode="grayscale")
        img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
        h_orig, w_orig, _ = img_array.shape

        for obj in root.findall('object'):
            cls_name = obj.find('name').text.strip().lower()
            try:
                cls_id = int(cls_name) + 1
            except ValueError:
                import string
                alphabet = list(string.ascii_lowercase)
                cls_id = alphabet.index(cls_name) + 1 if cls_name in alphabet else 1

            bndbox = obj.find('bndbox')
            if bndbox is None: continue

            xmin = float(bndbox.find('xmin').text)
            ymin = float(bndbox.find('ymin').text)
            xmax = float(bndbox.find('xmax').text)
            ymax = float(bndbox.find('ymax').text)

            # Convertiamo le coordinate se l'XML le fornisce normalizzate (0.0 - 1.0)
            if xmax <= 1.01 and w_orig > 1:
                xmin, xmax = xmin * w_orig, xmax * w_orig
                ymin, ymax = ymin * h_orig, ymax * h_orig

            # Calcolo del centro millimetrico della box
            cx = (xmin + xmax) / 2.0
            cy = (ymin + ymax) / 2.0

            # Calcoliamo l'ingombro reale della box per evitare sconfinamenti
            lato_box = max(int(xmax - xmin), int(ymax - ymin))
            if lato_box <= 4:
                lato_box = patch_size # Fallback di sicurezza

            half_lato = lato_box // 2

            # Coordinate di ritaglio sull'immagine originale
            x1 = max(0, int(cx - half_lato))
            x2 = min(w_orig, int(cx + half_lato))
            y1 = max(0, int(cy - half_lato))
            y2 = min(h_orig, int(cy + half_lato))

            if (x2 - x1) <= 1 or (y2 - y1) <= 1: continue

            # --- 1. RITAGLIO LETTERA ISOLATA ---
            crop_raw = img_array[y1:y2, x1:x2]
            crop_resized = tf.image.resize(crop_raw, [patch_size, patch_size]).numpy()

            window_images.append(crop_resized)
            window_classes.append(cls_id)
            window_confs.append(1.0)
            window_filenames.append(ann_file)

            # --- 2. RITAGLIO SFONDO PROPORZIONALE ---
            # Spostiamoci a destra esattamente di un'intera larghezza della box + un piccolo margine
            x1_bg = x1 + lato_box + 5
            if x1_bg + lato_box > w_orig:
                x1_bg = max(0, x1 - lato_box - 5) # Se non c'è spazio a destra, prova a sinistra
            x2_bg = x1_bg + lato_box

            crop_bg_raw = img_array[y1:y2, x1_bg:x2_bg]
            crop_bg_resized = tf.image.resize(crop_bg_raw, [patch_size, patch_size]).numpy()

            window_images.append(crop_bg_resized)
            window_classes.append(0)
            window_confs.append(0.0)
            window_filenames.append(ann_file)

    return np.array(window_images, dtype=np.float32), np.array(window_classes, dtype=np.int32), np.array(window_confs, dtype=np.float32), window_filenames

# Esecuzione del caricamento mirato a 48x48
DIMENSIONE_PATCH = 32
X_patches, y_patch_class, y_patch_conf, patch_filenames = load_narrow_patch_dataset_inspected(images_dir, annotations_dir, patch_size=DIMENSIONE_PATCH)
print(f"\n[RISULTATO] Ritagli totali estratti: {len(X_patches)}")

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split # <--- IMPORTANTE per evitare overfitting
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# ==========================================
# --- 4. PREPARAZIONE DATI (FIX OVERFITTING) ---
# ==========================================


# 1. Dividiamo in Train + Validation (85%) e Test (15%)
X_train_val, X_test, y_cls_train_val, y_cls_test, y_cnf_train_val, y_cnf_test = train_test_split(
    X_patches, y_patch_class, y_patch_conf, 
    test_size=0.15, 
    random_state=42
)

# 2. Dividiamo il Train_Val in Train (70% totale) e Validation (15% totale)
# Poiché 0.15 / 0.85 = ~0.176, usiamo test_size=0.176
X_train, X_val, y_cls_train, y_cls_val, y_cnf_train, y_cnf_val = train_test_split(
    X_train_val, y_cls_train_val, y_cnf_train_val, 
    test_size=0.176, 
    random_state=42
)

print(f"Split completato e MESCOLATO.")
print(f"Dataset suddiviso: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")


# ==========================================
# --- RESET DEL MODELLO DA ZERO ---
# ==========================================

# 1. Cancella i vecchi modelli rimasti nella memoria di Keras
tf.keras.backend.clear_session()

# 2. RE-INIZIALIZZA IL MODELLO (Sostituisci questa riga con la tua funzione reale!)
# Esempio: model = mia_funzione_che_definisce_la_rete()
# Se hai il codice della rete nel blocco prima, incollalo qui.
model = create_wide_separable_braille_net(num_classes=65) 


# ==========================================
# --- COMPILAZIONE ED ADDESTRAMENTO ---
# ==========================================
# 1. COMPILAZIONE CON PESI DELLE LOSS (Diamo peso 1.0 alla classe e solo 0.2 alla confidenza)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0), # Partiamo leggermente più alti (1e-3)
    loss={
        'final_class_output': tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        'final_conf_output': 'binary_crossentropy'
    },
    loss_weights={
        'final_class_output': 1.0,  # Focus totale sulla classificazione delle 65 classi
        'final_conf_output': 0.2    # Riduciamo l'impatto della confidenza sulla loss totale
    },
    metrics={
        'final_class_output': 'accuracy',
        'final_conf_output': 'accuracy'
    }
)

# 2. CALLBACKS PER MIGLIORARE L'ADDESTRAMENTO
callbacks_list = [
    # Se la val_loss della classe non migliora per 4 epoche, riduce il learning rate del 50%
    ReduceLROnPlateau(
        monitor='val_final_class_output_loss', 
        factor=0.5, 
        patience=4, 
        verbose=1, 
        min_lr=1e-6,
        mode='min'
    ),
    # Ferma l'addestramento se per 10 epoche non migliora più nulla (evita di perdere tempo)
    EarlyStopping(
        monitor='val_final_class_output_loss', 
        patience=10, 
        restore_best_weights=True, # Ti ricarica automaticamente i pesi migliori della storia
        mode='min'
    )
]

# 3. AVVIO FIT (Ora puoi aumentare le epoche a 60-80, si fermerà da solo grazie a EarlyStopping)
history = model.fit(
    X_train,
    {'final_class_output': y_cls_train, 'final_conf_output': y_cnf_train},
    validation_data=(X_val, {'final_class_output': y_cls_val, 'final_conf_output': y_cnf_val}),
    epochs=80, 
    batch_size=32,
    callbacks=callbacks_list,
    verbose=1
)

print("\nGenerazione dei grafici di addestramento in corso...")

# Definiamo il range delle epoche per l'asse X
epochs_range = range(1, len(history.history['loss']) + 1)

# Inizializziamo una figura con 1 riga e 3 colonne per metterli uno affianco all'altro
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

# --- 1. GRAFICO DELLE LOSS (Train e Val insieme per confronto diretto) ---
axs[0].plot(epochs_range, history.history['loss'], label='Total Loss (Train)', color='blue', linestyle='-')
axs[0].plot(epochs_range, history.history['val_loss'], label='Total Loss (Val)', color='blue', linestyle='--')
axs[0].plot(epochs_range, history.history['final_class_output_loss'], label='Class Loss (Train)', color='purple', linestyle='-')
axs[0].plot(epochs_range, history.history['val_final_class_output_loss'], label='Class Loss (Val)', color='purple', linestyle='--')
axs[0].plot(epochs_range, history.history['final_conf_output_loss'], label='Conf Loss (Train)', color='olive', linestyle='-')
# Se nel log originale non è presente 'val_final_conf_output_loss', si può commentare la riga seguente:
if 'val_final_conf_output_loss' in history.history:
    axs[0].plot(epochs_range, history.history['val_final_conf_output_loss'], label='Conf Loss (Val)', color='olive', linestyle='--')

axs[0].set_title('Andamento delle Loss', fontsize=12, fontweight='bold')
axs[0].set_xlabel('Epoche')
axs[0].set_ylabel('Valore Loss')
axs[0].legend(loc='upper right')
axs[0].grid(True, linestyle='--', alpha=0.6)

# --- 2. GRAFICO DELLE METRICHE DI ADDESTRAMENTO (TRAIN ACCURACY) ---
axs[1].plot(epochs_range, history.history['final_class_output_accuracy'], label='Class Accuracy', color='green', marker='o')
axs[1].plot(epochs_range, history.history['final_conf_output_accuracy'], label='Conf Accuracy', color='cyan', marker='s')
axs[1].set_title('Accuracies durante il TRAIN', fontsize=12, fontweight='bold')
axs[1].set_xlabel('Epoche')
axs[1].set_ylabel('Accuracy (0 - 1)')
axs[1].set_ylim(0, 1.05)
axs[1].legend(loc='lower right')
axs[1].grid(True, linestyle='--', alpha=0.6)

# --- 3. GRAFICO DELLE METRICHE DI VALIDAZIONE (VAL ACCURACY) ---
axs[2].plot(epochs_range, history.history['val_final_class_output_accuracy'], label='Val Class Accuracy', color='orange', marker='x')
axs[2].plot(epochs_range, history.history['val_final_conf_output_accuracy'], label='Val Conf Accuracy', color='magenta', marker='^')
axs[2].set_title('Accuracies durante la VALIDATION', fontsize=12, fontweight='bold')
axs[2].set_xlabel('Epoche')
axs[2].set_ylabel('Accuracy (0 - 1)')
axs[2].set_ylim(0, 1.05)
axs[2].legend(loc='lower right')
axs[2].grid(True, linestyle='--', alpha=0.6)

# Ottimizzazione del layout orizzontale e salvataggio/mostra dell'immagine
plt.tight_layout()
plt.savefig('training_performance_side_by_side.png', dpi=300) # Salva in formato panoramico ad alta risoluzione
plt.show()

print("Grafici affiancati salvati con successo nel file 'training_performance_side_by_side2.png'")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# =====================================================================
# 1. VALUTAZIONE GLOBALE DELLE LOSS E ACCURACY
# =====================================================================
print("\n" + "="*70)
print("   VALUTAZIONE DEL MODELLO SUL TEST SET (X_test)")
print("="*70)

# .evaluate restituisce la loss totale, le loss dei singoli output e le rispettive metriche
results = model.evaluate(
    X_test,
    {'final_class_output': y_cls_test, 'final_conf_output': y_cnf_test},
    batch_size=32,
    verbose=1
)

# Estraiamo i nomi delle metriche per stampare un report pulito
metric_names = model.metrics_names
print("\n--- Metriche Dettagliate dal Modello ---")
for name, value in zip(metric_names, results):
    print(f"  {name}: {value:.4f}")


# =====================================================================
# 2. GENERAZIONE DELLE PREDIZIONI E REPORT PER CLASSE
# =====================================================================
print("\n" + "="*70)
print("   ELABORAZIONE PREDIZIONI E CALCOLO METRICHE DI CLASSIFICAZIONE")
print("="*70)

# Generiamo le predizioni per tutto il test set
predictions = model.predict(X_test, batch_size=32, verbose=1)

# Il modello ha due output, quindi 'predictions' sarà una lista o un dizionario.
# Estraiamo i logit/probabilità della classificazione (65 classi)
# Nota: Keras di solito restituisce gli output nell'ordine in cui sono stati definiti o compilati.
if isinstance(predictions, list):
    pred_logits_class = predictions[0]  # Di solito il primo output è la classificazione
    pred_conf = predictions[1]
else:
    pred_logits_class = predictions['final_class_output']
    pred_conf = predictions['final_conf_output']

# Applichiamo argmax per ottenere la classe con la probabilità più alta
y_pred_classes = np.argmax(pred_logits_class, axis=1)

# Assicuriamoci che la ground truth sia in formato numerico/array 1D
y_true_classes = np.array(y_cls_test, dtype=np.int32)

# Calcolo accuratezza finale della classificazione
accuracy_finale = accuracy_score(y_true_classes, y_pred_classes)
print(f"\n[OK] ACCURATEZZA FINALE SULLA CLASSIFICAZIONE (65 Classi): {accuracy_finale * 100:.2f}%")

print("\n" + "-"*70)
print("REPORT DETTAGLIATO DI CLASSIFICAZIONE PER LE 65 CLASSI:")
print("-"*70)
# zero_division=0 evita scritte di errore/warning se alcune classi non sono state predette
print(classification_report(y_true_classes, y_pred_classes, zero_division=0))

# Generiamo la matrice di confusione
matrice_confusione = confusion_matrix(y_true_classes, y_pred_classes)
print("[OK] Matrice di confusione calcolata e salvata nella variabile 'matrice_confusione'.")


# =====================================================================
# 3. VERIFICA VISIVA SU UN CAMPIONE CASUALE
# =====================================================================
print("\n" + "="*70)
print("   TEST SU UN CAMPIONE CASUALE DEL TEST SET")
print("="*70)

idx_casuale = np.random.choice(len(X_test))
singolo_input = np.expand_dims(X_test[idx_casuale], axis=0)

# Predizione sul singolo elemento
singola_pred = model.predict(singolo_input, verbose=0)

if isinstance(singola_pred, list):
    singolo_output_class = singola_pred[0]
    singolo_output_conf = singola_pred[1]
else:
    singolo_output_class = singola_pred['final_class_output']
    singolo_output_conf = singola_pred['final_conf_output']

classe_predetta_singola = np.argmax(singolo_output_class[0])
classe_reale_singola = y_cls_test[idx_casuale]

# Calcolo della probabilità di presenza (sigmoide se l'output era lineare)
# Se hai usato l'attivazione 'sigmoid' nell'ultimo layer conf, puoi togliere la formula della sigmoide
prob_presenza = 1 / (1 + np.exp(-singolo_output_conf[0, 0])) 

print(f"Indice del campione testato: {idx_casuale}")
print(f"  Classe Reale: {classe_reale_singola}")
print(f"  Classe Predetta: {classe_predetta_singola}")
print(f"  Confidenza di Presenza: {prob_presenza:.4f}")
if classe_predetta_singola == classe_reale_singola:
    print("  -> RISULTATO: CORRETTO")
else:
    print("  -> RISULTATO: ERRATO")

In [ ]:
# =====================================================================
# 5. SALVATAGGIO
# =====================================================================
output_dir = os.path.join(extraction_path, 'output_modelli')
os.makedirs(output_dir, exist_ok=True)
keras_model_path = os.path.join(output_dir, 'tiny_braille_ssd_model_corretto.keras')
model.save(keras_model_path)
print(f"[OK] Modello salvato in: {keras_model_path}")

In [ ]:
print("\n" + "="*60)
print("   CONVERSIONE MODELLO PER EDGE AI: TFLITE FLOAT32 & model.h")
print("="*60)

# 1. Definiamo i nomi dei file di output richiesti
tflite_model_name = "braille_model_32x32_float32_leggero.tflite"
c_header_name = "braille_model_32x32_leggero.h"  # Modificato in model.h come richiesto

# 2. Conversione da Keras/SavedModel a TensorFlow Lite (Float32)
print(f"1. Avvio conversione del modello in formato TF Lite (FLOAT32)...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # Operazioni standard TF Lite
    tf.lite.OpsSet.SELECT_TF_OPS    # Operazioni avanzate TensorFlow (necessarie per SSD)
]

# === NOTA: La riga converter.optimizations = [tf.lite.Optimize.DEFAULT] è stata RIMOSTA ===
# Rimuovendola, disattiviamo la quantizzazione e forziamo il modello a rimanere in Float32 puro.

# Impediamo il crash dovuto a operazioni complesse non supportate nativamente
converter.experimental_new_converter = True

# Eseguiamo la conversione effettiva
tflite_model = converter.convert()

# Salviamo il file .tflite in Float32 su disco (nella memoria locale di Colab)
with open(tflite_model_name, "wb") as f:
    f.write(tflite_model)

tflite_size = os.path.getsize(tflite_model_name) / 1024
print(f"[OK] File TFLite Float32 generato con successo: '{tflite_model_name}'")
print(f"     Dimensione del file TFLite NON quantizzato: {tflite_size:.2f} KB")

# 3. Conversione da file .tflite a file sorgente C (model.h) per STM32
print(f"\n2. Generazione del file Header C ({c_header_name}) per STM32...")

def convert_tflite_to_c_header(tflite_bytes, header_path, array_name="g_braille_ssd_model_data"):
    with open(header_path, "w") as f:
        # Aggiornate le macro interne con il nome standard del file model.h
        f.write("#ifndef MODEL_H_\n")
        f.write("#define MODEL_H_\n\n")

        f.write("#include <stdint.h>\n\n")
        f.write(f"const unsigned int {array_name}_len = {len(tflite_bytes)};\n\n")
        f.write(f"const unsigned char {array_name}[] __attribute__((aligned(4))) = {{\n  ")

        for idx, byte in enumerate(tflite_bytes):
            f.write(f"0x{byte:02x}")
            if idx < len(tflite_bytes) - 1:
                f.write(", ")
            if (idx + 1) % 12 == 0 and idx < len(tflite_bytes) - 1:
                f.write("\n  ")

        f.write("\n};\n\n")
        f.write("#endif // MODEL_H_\n")

# Lanciamo la generazione del file model.h dai byte del modello Float32
convert_tflite_to_c_header(tflite_model, c_header_name)
print(f"[OK] File Header C '{c_header_name}' generato con successo!")

In [ ]:
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def test_tflite_model_on_memory_dataset(tflite_model_path, X_test_data, y_test_data, num_classes=65):
    """
    Inizializza l'interprete TF Lite ed esegue il calcolo delle metriche
    usando direttamente i dati X_test e y_test già presenti in memoria.
    """
    print(f"[INFO] Inizializzazione Interprete TF Lite da: {tflite_model_path}")

    # 1. ALLOCAZIONE DELL'INTERPRETE TF LITE
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    # Recuperiamo i dettagli di Input e Output richiesti dal modello TFLite
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_shape = input_details[0]['shape']
    print(f"[INFO] Il tuo modello TFLite accetta input con forma: {input_shape}")
    print(f"[INFO] Array di test ricevuto con forma: {X_test_data.shape}")

    y_true = np.array(y_test_data, dtype=np.int32)
    y_pred = []

    print("\nAvvio inferenza TF Lite su tutto il dataset X_test...")
    
    # 2. ESECUZIONE INFERENZA COMPLETA
    for i in tqdm(range(len(X_test_data))):
        single_input = X_test_data[i]
        
        # Gestione dimensioni: aggiunge dimensione batch se mancante
        if len(single_input.shape) == 3:
            single_input = np.expand_dims(single_input, axis=0)
            
        single_input = single_input.astype(input_details[0]['dtype'])

        interpreter.set_tensor(input_details[0]['index'], single_input)
        interpreter.invoke()

        # Recupero dei logit di output
        output_tensors_by_shape_dim = {}
        for output_detail in output_details:
            tensor_value = interpreter.get_tensor(output_detail['index'])
            output_tensors_by_shape_dim[tensor_value.shape[1]] = tensor_value

        pred_logits_tflite = None
        if num_classes in output_tensors_by_shape_dim:
            pred_logits_tflite = output_tensors_by_shape_dim[num_classes]

        if pred_logits_tflite is not None:
            classe_predetta = np.argmax(pred_logits_tflite[0])
            y_pred.append(classe_predetta)
        else:
            y_pred.append(-1) 

    y_pred = np.array(y_pred, dtype=np.int32)

    # 3. CALCOLO METRICHE FINALI
    accuracy = accuracy_score(y_true, y_pred)

    print("\n" + "="*70)
    print("   RISULTATI DEL TEST SUL DATASET DI TEST (X_test)")
    print("="*70)
    print(f"ACCURATEZZA GLOBALE (Accuracy): {accuracy * 100:.2f}%")
    print("-"*70)
    print("REPORT DETTAGLIATO PER CLASSE:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("="*70)

    conf_matrix = confusion_matrix(y_true, y_pred)
    
    # Restituiamo l'interprete configurato per usarlo nel blocco successivo senza ricaricarlo
    return interpreter, input_details, output_details, conf_matrix


# =====================================================================
# 1. CONFIGURAZIONE PERCORSI ED ESECUZIONE TEST COMPLETO
# =====================================================================
model_path = '/content/braille_model_32x32_float32_leggero.tflite'
NUM_CLASSES = 65

# Esegui il test globale su X_test e y_cls_test
interpreter, input_details, output_details, matrice_confusione = test_tflite_model_on_memory_dataset(
    tflite_model_path=model_path,
    X_test_data=X_test,      # La tua variabile in memoria
    y_test_data=y_cls_test,  # La tua variabile in memoria (Ground Truth)
    num_classes=NUM_CLASSES
)


# =====================================================================
# 2. TEST DETTAGLIATO SU UN SOTTOINSIEME CASUALE DI CAMPIONI
# =====================================================================
print("\n" + "="*70)
print("--- TEST SU UN SOTTOINSIEME CASUALE DEL TEST SET ---")
print("="*70)

# Controllo di sicurezza: se X_test ha meno di 1543 elementi, prende il massimo disponibile
num_samples_to_test = min(1543, len(X_test))
random_indices = np.random.choice(len(X_test), num_samples_to_test, replace=False)

for i, idx in enumerate(random_indices):
    sample_input_multi = np.expand_dims(X_test[idx], axis=0).astype(input_details[0]['dtype'])
    interpreter.set_tensor(input_details[0]['index'], sample_input_multi)
    interpreter.invoke()

    # Recupera i tensori di output
    output_tensors_by_shape_dim_multi = {}
    for output_detail_multi in output_details:
        tensor_value_multi = interpreter.get_tensor(output_detail_multi['index'])
        output_tensors_by_shape_dim_multi[tensor_value_multi.shape[1]] = tensor_value_multi

    pred_logits_tflite_multi = None
    pred_confs_tflite_multi = None

    # Verifica la presenza sia dei logit (es. 65) che della confidenza di presenza (es. 1)
    if NUM_CLASSES in output_tensors_by_shape_dim_multi and 1 in output_tensors_by_shape_dim_multi:
        pred_logits_tflite_multi = output_tensors_by_shape_dim_multi[NUM_CLASSES]
        pred_confs_tflite_multi = output_tensors_by_shape_dim_multi[1]
    else:
        print(f"[ERRORE] Impossibile recuperare gli output per il campione con indice {idx}")
        continue

    classe_predetta_multi = np.argmax(pred_logits_tflite_multi[0])
    
    # Calcolo della probabilità tramite sigmoide (per il valore di presenza)
    prob_presenza_multi = 1 / (1 + np.exp(-pred_confs_tflite_multi[0, 0]))
    classe_reale_multi = y_cls_test[idx]

    # Stampa i dettagli solo per i primi 10 campioni del sottogruppo per non intasare la console,
    # ma il calcolo viene comunque eseguito su tutti i campioni scelti.
    if i < 10:
        print(f"\nCampione {i+1} (Indice originale X_test: {idx}):")
        print(f"  Classe Reale: {classe_reale_multi}")
        print(f"  Classe Predetta da TFLite: {classe_predetta_multi}")
        print(f"  Probabilità di Presenza: {prob_presenza_multi:.4f}")
        if classe_predetta_multi == classe_reale_multi:
            print("  -> Previsione CORRETTA")
        else:
            print("  -> Previsione ERRATA")

print(f"\n[OK] Analisi comparativa completata con successo su {num_samples_to_test} campioni casuali.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("\n" + "="*75)
print("   TEST REALE AGGIORNATO: SEQUENZA PURA DELLE LETTERE (ORDINE DI LETTURA)")
print("="*75)

# 1. Peschiamo un indice casuale all'interno del Test Set
idx_scelto = np.random.randint(0, len(X_test))
file_corrispondente = test_filenames[idx_scelto]

print(f"Il ritaglio estratto appartiene al file: '{file_corrispondente}'")

# 2. Isoliamo SOLO i ritagli del file scelto che sono LETTERE REALI (conf_reale == 1.0)
# Questo elimina immediatamente i ritagli di sfondo duplicati che creavano confusione!
indici_lettere_nel_test = [
    i for i, f in enumerate(test_filenames)
    if f == file_corrispondente and y_cnf_test[i] == 1.0
]

num_lettere = len(indici_lettere_nel_test)
print(f"Nel Test Set sono state individuate {num_lettere} lettere reali per questo file.")

if num_lettere == 0:
    print("[AVVISO] Nessuna lettera reale trovata per questo file nel Test Set (forse sono finiti nello split di Train/Val). Riprova!")
else:
    # 3. Configurazione del grafico basato sulle lettere reali
    fig, axes = plt.subplots(1, num_lettere, figsize=(3.2 * num_lettere, 4.5))
    if num_lettere == 1:
        axes = [axes]

    giusti = 0

    for pos_grafica, idx_test in enumerate(indici_lettere_nel_test):
        ritaglio = X_test[idx_test]
        classe_reale = y_cls_test[idx_test]

        # Predizione del modello
        pred_logits, pred_confs = model.predict(np.expand_dims(ritaglio, axis=0), verbose=0)
        classe_predetta = np.argmax(pred_logits[0])
        prob_presenza = 1 / (1 + np.exp(-pred_confs[0, 0]))

        modello_vede_lettera = prob_presenza > 0.5

        stringa_gt = f"Reale: Lettera {classe_reale}"
        stringa_pred = f"Pred: Lettera {classe_predetta}\n({prob_presenza*100:.0f}%)" if modello_vede_lettera else f"Pred: Sfondo (0)\n({prob_presenza*100:.0f}%)"

        # Valutazione
        is_correct = (classe_predetta == classe_reale) and modello_vede_lettera
        if is_correct:
            giusti += 1
            colore_box = 'green'
        else:
            colore_box = 'red'

        # Plot
        ax = axes[pos_grafica]
        ax.imshow(ritaglio[:, :, 0], cmap='gray', vmin=0.0, vmax=1.0)
        ax.set_title(f"Lettera #{pos_grafica+1}\n{stringa_gt}\n{stringa_pred}", color=colore_box, fontsize=9, weight='bold')
        ax.axis('on')

    plt.suptitle(f"Parola: {file_corrispondente} | Lettere Ricostruite: {giusti}/{num_lettere} Corrette",
                 fontsize=12, weight='bold', y=1.08)
    plt.tight_layout()
    plt.show()